# Step 0. 환경 확인
내 맥북에서 AI 모델이 돌아가는지 확인합니다. 인터넷의 AI 서비스를 쓰지 않으므로 **API 키도, 요금도 없습니다.**

## 확인 항목
1. Ollama(올라마) 앱이 켜져 있는지
2. 실습에 쓸 모델 세 개 내려받기 (합쳐서 약 6.5GB)
3. Embedding(임베딩) 모델 호출
4. LLM 호출
5. 설정 파일 `config.json` 저장

> 먼저 https://ollama.com/download 에서 Ollama 앱을 설치하고 한 번 실행해 두세요. 메뉴 막대에 라마 아이콘이 보이면 됩니다.

## 실행하는 법
- 회색 상자(셀)를 클릭하고 **Shift + Enter**를 누르면 실행되고 다음 셀로 넘어갑니다.
- 왼쪽 `[ ]` 안이 `*`이면 실행 중, 숫자로 바뀌면 끝난 것입니다.
- 셀마다 `✅`가 나오면 다음 셀로 넘어가면 됩니다. `❌`가 나오면 그 아래 안내를 따라 주세요.
- 위에서부터 순서대로 실행해야 합니다.

In [ ]:
# 필요한 패키지를 설치합니다. 처음에는 1분쯤 걸립니다.
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "ollama"], check=True)
print("✅ 패키지 준비 완료. 다음 셀로 넘어가세요.")

## 설정
모델 이름을 바꾸고 싶으면 여기만 고치면 됩니다.

| 모델 | 역할 | 크기 |
|---|---|---|
| `bge-m3` | 글을 숫자(벡터)로 바꾸는 Embedding 모델. 한국어 지원 | 약 1.2GB |
| `gemma3:4b` | 문서를 읽고 답을 쓰는 LLM (Step 3). 한국어를 잘함 | 약 3.3GB |
| `llama3.2:3b` | 도구를 고르는 Agent용 LLM (Step 5). 작고 빠름 | 약 2GB |

> 메모리 16GB 이상인 맥이라면 `AGENT_MODEL`을 `gemma4:e4b`(약 9.5GB)로 바꾸면 도구를 더 정확히 씁니다.

In [ ]:
EMBED_MODEL = "bge-m3"
LLM_MODEL = "gemma3:4b"
AGENT_MODEL = "llama3.2:3b"   # 여유가 있으면 "gemma4:e4b"

## 1. Ollama 연결 확인

In [ ]:
import ollama

try:
    installed = [m.model for m in ollama.list().models]
    print("✅ Ollama 연결 성공")
    print("   설치된 모델:", ", ".join(installed) if installed else "(없음)")
except Exception as e:
    installed = []
    print(f"❌ Ollama에 연결하지 못했습니다: {e}")
    print("   → 응용 프로그램에서 Ollama 앱을 실행한 뒤 이 셀을 다시 실행해 주세요.")

## 2. 모델 내려받기
처음 한 번만 받습니다. 인터넷 속도에 따라 몇 분 걸립니다. 이미 있으면 바로 넘어갑니다.

In [ ]:
for name in [EMBED_MODEL, LLM_MODEL, AGENT_MODEL]:
    if any(m == name or m.startswith(name + ":") for m in installed):
        print(f"✅ {name} 이미 있음")
        continue
    print(f"⬇️  {name} 내려받는 중... (잠시 기다려 주세요)")
    ollama.pull(name)
    print(f"✅ {name} 내려받기 완료")

## 3. Embedding 모델 호출 확인
문장 하나를 넣으면 숫자 1,024개가 나옵니다. 이 숫자 묶음이 문장의 "좌표"입니다.

In [ ]:
try:
    vec = ollama.embed(model=EMBED_MODEL, input="테스트 문장입니다.")["embeddings"][0]
    print("✅ Embedding 호출 성공")
    print(f"   차원 수: {len(vec)}")
    print(f"   앞의 5개: {[round(v, 4) for v in vec[:5]]}")
except Exception as e:
    print(f"❌ Embedding 호출 실패: {e}")

## 4. LLM 호출 확인
모델을 처음 부를 때는 메모리에 올리느라 몇 초 더 걸립니다.

In [ ]:
for name in [LLM_MODEL, AGENT_MODEL]:
    try:
        r = ollama.chat(model=name, messages=[{"role": "user", "content": "안녕? 한 줄로 대답해 줘."}])
        print(f"✅ {name} 호출 성공")
        print("   답:", r["message"]["content"].strip())
    except Exception as e:
        print(f"❌ {name} 호출 실패: {e}")

## 5. 설정 저장
다음 Step의 노트북들이 이 파일을 읽습니다.

In [ ]:
import json

with open("config.json", "w") as f:
    json.dump({"EMBED_MODEL": EMBED_MODEL, "LLM_MODEL": LLM_MODEL, "AGENT_MODEL": AGENT_MODEL}, f, indent=2)
print("✅ config.json 저장 완료. Step 1로 넘어가세요.")